# Downstream tables — Renly-era vs Sep-23 rebuild

The tables below are built from `OpenAlex/output` (pcs, PPP, Atypicality builders and caches). Their Renly-era versions
were copied to `<area>/Old/pre_sep23_2026-10-08/` on 2026-10-08 before the reruns; this notebook compares each file with
its rebuild:

* rows, and per column the null share and mean in both versions (schema changes are listed);
* the same rows in both versions, joined on the table's key (a fixed hash sample when the table is large): share equal,
  mean shift, and Pearson/Spearman correlation per numeric column.

Output: `validation/data/sep23_downstream_columns.csv`, `validation/data/sep23_downstream_paired.csv`.

In [1]:
import os, time
import numpy as np, pandas as pd, duckdb, pyarrow.parquet as pq

SOS = '/project/jevans/Dawoon/Science of Science'
B = 'Old/pre_sep23_2026-10-08'
OUTDIR = f'{SOS}/validation/data'
TMP = f'{SOS}/OpenAlex/cache/duckdb_tmp_downstream_comparison'; os.makedirs(TMP, exist_ok=True)
con = duckdb.connect()
con.execute(f"SET threads={os.environ.get('SLURM_CPUS_PER_TASK', '8')}; SET memory_limit='{os.environ.get('NB_DUCKDB_MEM', '100GB')}'; "
            f"SET preserve_insertion_order=false; SET temp_directory='{TMP}'; SET enable_progress_bar=false")
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 400)

# (label, new path relative to SOS, old path relative to SOS, key columns or None)
A = 'Atypicality'
FILES = [
    ('pcs_citation',            'pcs/output/pcs_citation.parquet',            f'pcs/{B}/output/pcs_citation.parquet',            ['paper_id']),
    ('pcs_citation_trend',      'pcs/output/pcs_citation_trend.parquet',      f'pcs/{B}/output/pcs_citation_trend.parquet',      ['paper_id', 'cite_year']),
    ('pcs_hit_probability',     'pcs/output/pcs_hit_probability.parquet',     f'pcs/{B}/output/pcs_hit_probability.parquet',     ['paper_id']),
    ('ppp_paper_trend',         'PPP/output/ppp_paper_trend.parquet',         f'PPP/{B}/output/ppp_paper_trend.parquet',         ['paperid', 'patent', 'cite_year']),
    ('ppp_patent_trend',        'PPP/output/ppp_patent_trend.parquet',        f'PPP/{B}/output/ppp_patent_trend.parquet',        ['paperid', 'patent', 'cite_year']),
    ('ppp_master_rev',          f'{A}/Data/ppp_master_rev.parquet',           f'{A}/{B}/Data/ppp_master_rev.parquet',           ['paper_id', 'patent_id']),
    ('ppp_master_rev_pppl',     f'{A}/Data/ppp_master_rev_pppl.parquet',      f'{A}/{B}/Data/ppp_master_rev_pppl.parquet',      ['paper_id', 'patent_id']),
    ('paper_master (rev)',      f'{A}/Data/rev/paper_master.parquet',         f'{A}/{B}/Data/rev/paper_master.parquet',         ['paper_id']),
    ('curvature_paper (rev)',   f'{A}/Data/rev/curvature_paper.parquet',      f'{A}/{B}/Data/rev/curvature_paper.parquet',      ['doc_id']),
    ('paper_year_DA cache',     f'{A}/Data/_traj_cache/paper_year_DA/all_years/paper_metrics.parquet', f'{A}/{B}/Data/_traj_cache/paper_year_DA/all_years/paper_metrics.parquet', None),
    ('paper2000_DA_metrics',    f'{A}/Data/_traj_cache/paper2000_DA_metrics.parquet', f'{A}/{B}/Data/_traj_cache/paper2000_DA_metrics.parquet', None),
    ('paper2020_DA_metrics',    f'{A}/Data/_traj_cache/paper2020_DA_metrics.parquet', f'{A}/{B}/Data/_traj_cache/paper2020_DA_metrics.parquet', None),
    ('rev paper_disruption cache', f'{A}/Data/rev/_traj_cache/paper_disruption.parquet', f'{A}/{B}/Data/rev/_traj_cache/paper_disruption.parquet', None),
    ('rev curv_metrics_paper_h10', f'{A}/Data/rev/_traj_cache/curv_metrics_paper_h10.parquet', f'{A}/{B}/Data/rev/_traj_cache/curv_metrics_paper_h10.parquet', None),
    ('rev citation_age_docs_paper_h20', f'{A}/Data/rev/_traj_cache/citation_age_docs_paper_h20.parquet', f'{A}/{B}/Data/rev/_traj_cache/citation_age_docs_paper_h20.parquet', None),
    ('citation_trajectory papers_h10', f'{A}/Data/_traj_cache/citation_trajectory/papers_h10.parquet', f'{A}/{B}/Data/_traj_cache/citation_trajectory/papers_h10.parquet', ['doc_id', 'source']),
]
EXTRA = os.environ.get('NB_EXTRA_FILES')            # optional "label|new|old|key1,key2;..." additions
if EXTRA:
    for item in EXTRA.split(';'):
        lab, n, o, k = item.split('|'); FILES.append((lab, n, o, k.split(',') if k else None))
GUESS_KEYS = [['paper_id', 'patent_id'], ['paperid', 'patent', 'cite_year'], ['paper_id', 'cite_year'], ['paper_id'], ['doc_id'], ['work_id']]
SAMPLE_ROWS = 4_000_000
t00 = time.time()

## 1. Rows and columns

In [2]:
def schema(p):
    return pq.ParquetFile(p).schema_arrow

col_rows, have = [], []
for lab, n, o, key in FILES:
    pn, po = f'{SOS}/{n}', f'{SOS}/{o}'
    if not (os.path.exists(pn) and os.path.exists(po)):
        print(f'{lab:<34} skipped (new: {os.path.exists(pn)}, old: {os.path.exists(po)})'); continue
    sn, so = schema(pn), schema(po)
    rn, ro = pq.ParquetFile(pn).metadata.num_rows, pq.ParquetFile(po).metadata.num_rows
    added, dropped = sorted(set(sn.names) - set(so.names)), sorted(set(so.names) - set(sn.names))
    if key is None:
        key = next((k for k in GUESS_KEYS if all(c in sn.names and c in so.names for c in k)), None)
    have.append((lab, pn, po, key))
    print(f'{lab:<34} rows {ro:>13,} -> {rn:>13,} ({(rn / ro - 1) * 100 if ro else float("nan"):+6.1f}%)  key {key}'
          + (f'  added {added}' if added else '') + (f'  dropped {dropped}' if dropped else ''))
    common = [f for f in sn if f.name in so.names]
    for f in common:
        t = str(f.type)
        isnum = any(x in t for x in ('int', 'float', 'double', 'decimal')) and not t.startswith('list')
        agg = ', '.join([f'count("{f.name}")'] + ([f'avg("{f.name}")'] if isnum else []))
        a = con.execute(f"SELECT count(*), {agg} FROM read_parquet('{po}')").fetchone()
        b = con.execute(f"SELECT count(*), {agg} FROM read_parquet('{pn}')").fetchone()
        col_rows.append(dict(table=lab, column=f.name, type=t, null_old_pct=(1 - a[1] / a[0]) * 100 if a[0] else np.nan,
                             null_new_pct=(1 - b[1] / b[0]) * 100 if b[0] else np.nan,
                             mean_old=a[2] if isnum else np.nan, mean_new=b[2] if isnum else np.nan))
cols = pd.DataFrame(col_rows)
cols.to_csv(f'{OUTDIR}/sep23_downstream_columns.csv', index=False)
moved = cols[(cols.null_old_pct - cols.null_new_pct).abs().gt(1) | ((cols.mean_new / cols.mean_old - 1).abs().gt(0.05))]
print(f'\ncolumns whose null share moved > 1 point or mean > 5% ({len(moved)} of {len(cols)}):')
print(moved.round(4).to_string(index=False))

pcs_citation                       rows     5,106,527 ->     5,106,527 (  +0.0%)  key ['paper_id']
pcs_citation_trend                 rows    12,992,507 ->    14,498,485 ( +11.6%)  key ['paper_id', 'cite_year']
pcs_hit_probability                rows     4,480,434 ->     5,022,564 ( +12.1%)  key ['paper_id']
ppp_paper_trend                    rows     5,786,928 ->     6,724,083 ( +16.2%)  key ['paperid', 'patent', 'cite_year']
ppp_patent_trend                   rows     2,906,165 ->     2,906,165 (  +0.0%)  key ['paperid', 'patent', 'cite_year']
ppp_master_rev                     rows       488,894 ->       488,894 (  +0.0%)  key ['paper_id', 'patent_id']
ppp_master_rev_pppl                rows       488,894 ->       488,894 (  +0.0%)  key ['paper_id', 'patent_id']
paper_master (rev)                 rows       123,000 ->       123,000 (  +0.0%)  key ['paper_id']
curvature_paper (rev)              rows   126,628,527 ->   161,799,449 ( +27.8%)  key ['doc_id']
paper_year_DA cache         

## 2. The same rows in both versions

In [3]:
pair_rows = []
for lab, pn, po, key in have:
    if not key:
        print(f'{lab}: no key, unpaired'); continue
    sn = schema(pn); so = schema(po)
    nums = [f.name for f in sn if f.name in so.names and f.name not in key and not str(f.type).startswith('list')
            and any(x in str(f.type) for x in ('int', 'float', 'double'))]
    n_rows = pq.ParquetFile(pn).metadata.num_rows
    k = max(1, n_rows // SAMPLE_ROWS)
    keyexpr = ' || \'|\' || '.join(f'CAST("{c}" AS VARCHAR)' for c in key)
    where = f"WHERE hash({keyexpr}) % {k} = 0" if k > 1 else ''
    sel = ', '.join([f'{keyexpr} AS k'] + [f'"{c}"' for c in nums])
    o = con.execute(f"SELECT {sel} FROM read_parquet('{po}') {where}").df()
    n = con.execute(f"SELECT {sel} FROM read_parquet('{pn}') {where}").df()
    dup = int(o.k.duplicated().sum() + n.k.duplicated().sum())
    o, n = o.drop_duplicates('k'), n.drop_duplicates('k')
    j = o.merge(n, on='k', suffixes=('_old', '_new'))
    print(f'{lab:<34} sample 1/{k}: old {len(o):,}, new {len(n):,}, both {len(j):,}, old only {len(set(o.k) - set(n.k)):,}, new only {len(set(n.k) - set(o.k)):,}'
          + (f'  ({dup} duplicate keys dropped)' if dup else ''))
    for c in nums:
        a, b = j[f'{c}_old'].astype(float), j[f'{c}_new'].astype(float)
        ok = a.notna() & b.notna()
        if ok.sum() < 3:
            continue
        a, b = a[ok], b[ok]
        pair_rows.append(dict(table=lab, column=c, n=int(ok.sum()), equal_pct=float((a == b).mean() * 100),
                              mean_old=a.mean(), mean_new=b.mean(),
                              pearson=float(np.corrcoef(a, b)[0, 1]) if a.std() and b.std() else np.nan,
                              spearman=float(a.rank().corr(b.rank())) if a.std() and b.std() else np.nan))
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f'{OUTDIR}/sep23_downstream_paired.csv', index=False)
print('\nleast stable columns (lowest share equal):')
print(pairs.sort_values('equal_pct').head(60).round(4).to_string(index=False))
print(f'\ndone in {(time.time() - t00) / 60:.1f} min')

pcs_citation                       sample 1/1: old 5,106,527, new 5,106,527, both 5,106,527, old only 0, new only 0
pcs_citation_trend                 sample 1/3: old 4,331,380, new 4,833,289, both 4,322,719, old only 8,661, new only 510,570
pcs_hit_probability                sample 1/1: old 4,480,434, new 5,022,564, both 4,470,369, old only 10,065, new only 552,195
ppp_paper_trend                    sample 1/1: old 5,786,928, new 6,724,083, both 5,779,162, old only 7,766, new only 944,921
ppp_patent_trend                   sample 1/1: old 2,906,165, new 2,906,165, both 2,906,165, old only 0, new only 0
ppp_master_rev                     sample 1/1: old 488,894, new 488,894, both 488,894, old only 0, new only 0
ppp_master_rev_pppl                sample 1/1: old 488,894, new 488,894, both 488,894, old only 0, new only 0
paper_master (rev)                 sample 1/1: old 123,000, new 123,000, both 123,000, old only 0, new only 0
curvature_paper (rev)              sample 1/40: old 3,166,1